In [10]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('farm_train.csv')
test = pd.read_csv('farm_test.csv')

print("train :", train.shape)
print("test :", test.shape)

train : (4000, 9)
test : (1000, 8)


In [3]:
train.head()

,농업면적,연도,지역,비료사용량,비료잔여량,작물종류,토양유형,농약검출여부,등급
0,20079.652837,2004,대구,407.985516,146.290507,보리,점토,2,C
1,73858.643204,2012,울산,221.229692,1967.333638,밀,점토,0,B
2,65718.150861,2012,강원,370.967205,2253.522610,쌀,점토,0,B
3,37366.182902,2005,광주,274.128236,1487.535265,쌀,양토,0,B
4,81515.151289,2007,충남,213.410655,683.306745,쌀,양토,1,B


In [4]:
test.head()

,농업면적,연도,지역,비료사용량,비료잔여량,작물종류,토양유형,등급
0,43284.730127,2008,울산,138.767165,415.437269,보리,점토,B
1,97624.943779,2014,충북,373.560368,2809.120053,보리,양토,A
2,47115.257674,2023,대전,249.082661,1586.452521,쌀,점토,B
3,82817.237255,2012,충북,242.284660,142.494412,쌀,점토,C
4,73866.403857,2012,광주,166.180358,4365.543337,밀,점토,A


In [5]:
target = train.pop('농약검출여부')

In [7]:
data = pd.concat([train,test])
data = pd.get_dummies(data)
print("data :", data.shape)

data : (5000, 28)


In [8]:
train = data.iloc[:4000]
test = data.iloc[4000:]

In [11]:
from sklearn.model_selection import train_test_split

In [13]:
#불균형데이터
x_train,x_val,y_train,y_val = train_test_split(
    train,
    target,
    test_size=0.2,
    random_state=0,
    stratify=target,
)
print("x_train :", x_train.shape)
print("x_val :", x_val.shape)
print("y_train :", y_train.shape)
print("y_val :", y_val.shape)

x_train : (3200, 28)
x_val : (800, 28)
y_train : (3200,)
y_val : (800,)


In [14]:
from sklearn.ensemble import RandomForestClassifier

In [15]:
model1 = RandomForestClassifier(random_state=0, class_weight='balanced')
model1.fit(x_train,y_train)
pred1 = model1.predict(x_val)

In [16]:
from sklearn.metrics import f1_score

In [19]:
f1 = f1_score(y_val,pred1, average='macro')
print("랜덤포레스트 :", f1)

랜덤포레스트 : 0.8113464700206591


In [20]:
from lightgbm import LGBMClassifier

In [22]:
model2 = LGBMClassifier(random_state=0, verbose=-1, class_weight='balanced')
model2.fit(x_train,y_train)
pred2 = model2.predict(x_val)

In [24]:
f1 = f1_score(y_val,pred2, average='macro')
print("LGBM :", f1)

LGBM : 0.9056417097662033


In [26]:
pred = model2.predict(test)

result = pd.DataFrame({
    'pred' : pred
})

result.to_csv('result.csv', index=False)

In [27]:
result.head()

,pred
0,2
1,0
2,0
3,2
4,0
